# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def window(x):
    return src.utils.get_windowed(x, stride=120)


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### most data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### "wide" subsurface data

In [ ]:
## should we use "wide" data?
USE_WIDE = True

if USE_WIDE:

    ## load spatial data
    forced_wide, anom_wide = load_consolidated_wide()

    for v in list(forced_wide):
        forced[v] = forced_wide[v]
        anom[v] = anom_wide[v]

#### max grad thermocline

In [ ]:
h_mg_forced, h_mg = src.utils.load_h_data(max_grad=True)

#### ELI

In [ ]:
pt_fp = pathlib.Path(os.environ["SAVE_FP"], f"eli_scaled_Tw.nc")
# pt_fp = pathlib.Path(os.environ["SAVE_FP"], f"eli_05_scaled_Tw_updated.nc")
eli_pt = xr.open_dataarray(pt_fp)

### scaling for mean thermocline depth

#### Mean thermocline depth

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

## Compute/plot

### Specify variabes

In [ ]:
## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
# LONS_W = dict(longitude=slice(120, 160))
LONS_W = dict(longitude=slice(120, 180))
# LONS_W = dict(longitude=slice(120, 210))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
LON_AVG_34 = lambda x: x.sel(longitude=slice(190, 240)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=slice(50, 80)).mean("z_t")
ML_AVG = lambda x, Hm=50: x.sel(z_t=slice(None, Hm)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x, Hm=50: ML_AVG(LON_AVG(x), Hm=Hm)
T34_ML_AVG = lambda x, Hm=50: ML_AVG(LON_AVG_34(x), Hm=Hm)

anom["T_3_ml"] = src.utils.reconstruct_wrapper(
    anom[["T", "T_comp"]],
    fn=T3_ML_AVG,
)["T"]

anom["T_34_ml"] = src.utils.reconstruct_wrapper(
    anom[["T", "T_comp"]],
    fn=T34_ML_AVG,
)["T"]

anom["T_3_ml_30"] = src.utils.reconstruct_wrapper(
    anom[["T", "T_comp"]],
    fn=lambda x: T3_ML_AVG(x, Hm=30),
)["T"]

anom["T_3_ml_20"] = src.utils.reconstruct_wrapper(
    anom[["T", "T_comp"]],
    fn=lambda x: T3_ML_AVG(x, Hm=20),
)["T"]

In [ ]:
## should we use OHC
USE_OHC = True

lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

if USE_OHC:

    ## compute ohc
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W),
    )["T"]

else:
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_W)),
    )["ssh"]

#### helper funcs

In [ ]:
def regress_over_time(data, x_vars, y_vars):
    """regression over time"""

    ## get windowed data
    data_ = window(data)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars)

    ## loop thru years
    for year in tqdm.tqdm(data_.year):

        ## get grouped data
        data_y = data_.sel(year=year).groupby("time.month")

        ## do regression
        coefs.append(data_y.map(src.utils.regress_xr_proj, **kwargs))

    return xr.concat(coefs, dim=data_.year)


def regress_wrapper(data, x_vars, y_var, y_fn):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var])


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def make_scatter_ax(ax, anom_, xvar, yvar, month):
    """scatter plot of data for given month"""

    ## prep func
    get_season = lambda x: src.utils.sel_month(
        x.resample({"time": "QS-DEC"}).mean(), month
    )
    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)

    ## plot data
    ax.scatter(*plot_data, s=3, label=f"cov = {cov.item():.1e}")

    ax.set_title(f"corr = {corr.item():.2f}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month):
    """scatter plot of data for given month"""

    ## prep func
    get_season = lambda x: src.utils.sel_month(
        x.resample({"time": "QS-DEC"}).mean(), month
    )
    prep = lambda x: get_season(x).transpose("time", "member")

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

    for ax, t_idx in zip(axs, [["1850", "1889"], ["1950", "1989"], ["2061", "2100"]]):

        ## helper func
        prep_ = lambda x: prep(x).sel(time=slice(*t_idx))

        ## scatter plot of data
        ax = make_scatter_ax(
            ax,
            prep_(anom_),
            xvar=xvar,
            yvar=yvar,
            month=month,
        )

    ## format/scale axes
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[2].set_yticks([])

    return fig, axs

### Bulk estimate (like RO)

In [ ]:
def regress_harm(data, x_vars, y_vars, max_order=3):
    """regression over time"""

    ## get windowed data
    # data_ = window(data)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(max_order=max_order, x_vars=x_vars, y_vars=y_vars)

    ## loop thru years
    for year in tqdm.tqdm(data.year):

        coefs.append(src.utils.regress_harm_wrapper(data.sel(year=year), **kwargs))

    return xr.concat(coefs, dim=data.year)

#### estimate with finite differences

In [ ]:
## specify variables
# varnames = ["eli_05_scaled","h_w"]
varnames = ["T_3_ml", "h_w"]  # , "h_w_sqr"]
v0 = varnames[0]
v1 = varnames[1]
ddt_varnames = [f"ddt_{v0}", f"ddt_{v1}"]
# ddt_varnames = [f"ddt_T_3_ml", f"ddt_{v1}"]

## merge data
Th_ = anom[["T_3", "T_34", "T_34_ml", "T_3_ml", "T_3_ml_30", "h_w", "T_3_ml_20"]]
Th_ = window(Th_.sel(time=slice("1851", None)))
Th_ = xr.merge([Th_, eli_pt])

## normalize
Th_normed = Th_ / Th_.std()
Th_normed["h_w_sqr"] = Th_normed["h_w"] ** 2

## differentiate
ddt_Th = src.utils.get_ddt(Th_normed, is_forward=True)

In [ ]:
## compute operator
L = (
    regress_harm(ddt_Th, x_vars=varnames, y_vars=ddt_varnames, max_order=3)
    .squeeze(drop=True)
    .rename({"ell": "j"})
)

## convert to dataarray
L = L.to_dataarray(dim="y").rename({"j": "x"})

## extract data
R = L.isel(y=0, x=0)
eps = -L.isel(y=1, x=1)
F1 = L.isel(y=0, x=1)
F2 = -L.isel(y=1, x=0)

## look at residuals
Y = ddt_Th[ddt_varnames].to_dataarray(dim="y")
X = ddt_Th[varnames].to_dataarray(dim="x")
Yhat = (X.groupby("time.month") * L).sum("x")

## compute stats
get_corr = lambda x: xr.corr(x["Y"], x["Yhat"], dim=["member", "time"])
results = xr.merge([Y.rename("Y"), Yhat.rename("Yhat")])
corr = results.groupby("time.month").map(get_corr)
error = np.sqrt((Yhat - Y) ** 2).groupby("time.month").mean(["member", "time"])

#### params

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))
for ax, p in zip(axs, [R, -eps, R - eps]):
    ax.plot(R.month, p.sel(year=1871))
    ax.plot(R.month, p.sel(year=1971))
    ax.plot(R.month, p.sel(year=2021))
    ax.plot(R.month, p.sel(year=2081))
    ax.axvspan(2.8, 5.2, alpha=0.2)
src.utils.set_lims(axs)
plt.show()
# plt.plot(eps.isel(year=0))

fig, axs = plt.subplots(1, 4, figsize=(8, 2), layout="constrained")
axs[0].plot(eps.year, (R).mean("month"))
axs[1].plot(eps.year, (eps).mean("month"))
axs[2].plot(eps.year, (R - eps).mean("month"))
axs[3].plot(eps.year, np.sqrt(F1.mean("month") * F2.mean("month")))
axs[0].set_title("R")
axs[1].set_title("eps")
axs[2].set_title("R-eps")
axs[3].set_title("Wyrtki")
axs[0].axvline(2020, c="k")
src.utils.set_lims(axs[:2])
plt.show()

#### signal-to-noise

In [ ]:
## specify which variable to look at
yn = 0

print("corr")
fig, axs = plt.subplots(1, 2, figsize=(5, 2.5))
axs[0].plot(corr.year, corr.mean("month").isel(y=yn))
axs[1].plot(corr.month, corr.sel(year=1871).isel(y=yn))
axs[1].plot(corr.month, corr.sel(year=1971).isel(y=yn))
axs[1].plot(corr.month, corr.sel(year=2021).isel(y=yn))
axs[1].plot(corr.month, corr.sel(year=2081).isel(y=yn))
plt.show()

print("error")
fig, axs = plt.subplots(1, 2, figsize=(5, 2.5))
axs[0].plot(error.year, error.mean("month").isel(y=yn))
axs[1].plot(error.month, error.sel(year=1871).isel(y=yn))
axs[1].plot(error.month, error.sel(year=1971).isel(y=yn))
axs[1].plot(error.month, error.sel(year=2021).isel(y=yn))
axs[1].plot(error.month, error.sel(year=2081).isel(y=yn))
axs[1].axvline(6)
plt.show()

#### noise

In [ ]:
sel = lambda x: src.utils.sel_month(x, 4)
# sel = lambda x : x
X6 = sel(X).transpose("time", "member", ...)
res_6 = sel(Y).isel(y=0).transpose("time", "member", ...)
# res_6 = sel(Y-Yhat).isel(y=0).transpose("time","member",...)

xi = 1
sel0 = lambda x: x.sel(year=2001)
sel1 = lambda x: x.sel(year=2081)

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5))

axs[0].scatter(sel0(X6.isel(x=xi)), sel0(res_6), s=2, alpha=0.5)

axs[1].scatter(sel1(X6.isel(x=xi)), sel1(res_6), s=2, alpha=0.5)

src.utils.set_lims(axs)
plt.show()

## Level 2: look at dominant terms

### mean upwelling

In [ ]:
## specify mixed layer depth
H0 = 50

## get w_bar
w_bar = src.utils.reconstruct_clim(window(forced_wide[["w", "w_comp"]]))["w"]
is_downwelling = w_bar < 0

## only keep upwelling
w_bar_upwelling = w_bar.where(~(is_downwelling), other=0.0)

## average over Niño 3 and scale by MLD
w_H = 1 / H0 * T3_ENT_AVG(w_bar_upwelling).rename("w_H")
# w_H = 1 / H0 * w_bar_upwelling.sel(z_t=50, method="nearest").rename("w_H")

### mean stratification

In [ ]:
## specify mixed layer depth
H0 = 50

## get Tbar
T_bar = src.utils.reconstruct_clim(window(forced_wide[["T", "T_comp"]]))["T"]

## get dTbar_dz
dTdz_bar = 1 / H0 * (ENT_AVG(T_bar) - ML_AVG(T_bar))
dTdz_bar_v2 = ML_AVG(T_bar.differentiate("z_t"))

## only keep upwelling
is_downwelling = ENT_AVG(w_bar) < 0
dTdz_bar_upwelling = dTdz_bar.where(~(is_downwelling), other=0.0)
dTdz_bar_upwelling_v2 = dTdz_bar_v2.where(~(is_downwelling), other=0.0)

## average over Niño 3
dTdz_bar_T3 = LON_AVG(dTdz_bar_upwelling).rename("dTdz_bar")

## compute anomalies

In [ ]:
## indices
Tsub = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    # fn=T3_ENT_AVG
    fn=lambda x: T3_ENT_AVG(x) - T3_ML_AVG(x),
)["T"]
w = src.utils.reconstruct_wrapper(anom_wide[["w", "w_comp"]], fn=T3_ENT_AVG)["w"]

## thermocline feedback
anom_ = xr.merge([Tsub, anom[["T_3_ml", "h_w"]]])
thf = w_H * regress_over_time(anom_, x_vars=["T_3_ml", "h_w"], y_vars=["T"])

## ekman feedback
anom_ = xr.merge([w, anom[["T_3_ml", "h_w"]]])
ekf = dTdz_bar_T3 * regress_over_time(anom_, x_vars=["T_3_ml", "h_w"], y_vars=["w"])

## extract data
ekf = ekf["w"]
thf = thf["T"]

### compute zonal terms

#### ZAF

In [ ]:
## temperature clim
Tbar = src.utils.reconstruct_clim(
    window(forced_wide[["T", "T_comp"]]), fn=lambda x: x.sel(z_t=slice(None, 50))
)

## anomalies
data_ = xr.merge([anom_wide[["u"]], anom[["T_3_ml", "h_w"]]])

## regress
u_coefs = regress_over_time(data=data_, x_vars=["T_3_ml", "h_w"], y_vars=["u"])

## reconstruct
u_coefs = src.utils.reconstruct_fn(
    scores=u_coefs["u"],
    components=anom_wide["u_comp"],
    fn=lambda x: x.sel(z_t=slice(None, 50)),
)

## compute ZAF, and convert to 1/year
zaf = src.utils.get_ZAF(bar=Tbar, prime=u_coefs.to_dataset(name="u"))
zaf = T3_ML_AVG(zaf)

#### DD

In [ ]:
## temperature clim
ubar = src.utils.reconstruct_clim(
    window(forced_wide[["u", "u_comp"]]), fn=lambda x: x.sel(z_t=slice(None, 50))
)

## anomalies
data_ = xr.merge([anom_wide[["T"]], anom[["T_3_ml", "h_w"]]])

## regress
T_coefs = regress_over_time(data=data_, x_vars=["T_3_ml", "h_w"], y_vars=["T"])

## reconstruct
T_coefs = src.utils.reconstruct_fn(
    scores=T_coefs["T"],
    components=anom_wide["T_comp"],
    fn=lambda x: x.sel(z_t=slice(None, 50)),
)

## compute ZAF, and convert to 1/year
dd = src.utils.get_DD(bar=ubar, prime=T_coefs.to_dataset(name="T"))
dd = T3_ML_AVG(dd)

#### $Q$

In [ ]:
## compute nhf (W/m2)
nhf = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3)[
    "nhf"
]

## convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
rho = 1.02e3
Cp = 4.2e3
Q = nhf * sec_per_mo / (rho * Cp * H0)

In [ ]:
## regress over time
anom_ = xr.merge([Q.rename("Q"), anom[["T_3", "T_34"]]])
alpha = regress_over_time(anom_, x_vars=["T_3"], y_vars=["Q"])
alpha = alpha["Q"].squeeze(drop=True)

#### Plot results

In [ ]:
delta = lambda x: x - x.isel(year=0)
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(alpha.year, delta(12 * alpha.mean("month")), label=r"$-\alpha$")
ax.plot(alpha.year, delta(12 * thf.isel(j=0).mean("month")), label="THF")
ax.plot(alpha.year, delta(12 * ekf.isel(j=0).mean("month")), label="EKF")
# ax.plot(alpha.year, delta(12*(zaf+dd).isel(j=0).mean("month")), label=r"$u\cdot \partial T/\partial x$")
ax.plot(R.year, delta((R - eps).mean("month")), c="k")
# ax.plot(R.year, delta((R-eps).mean("month")), c="k")
ax.axvline(2020, c="k", ls="--", lw=0.8)
ax.axhline(0, c="k", ls="--", lw=0.8)
ax.legend()
plt.show()

In [ ]:
ji = 0

fig, axs = plt.subplots(1, 5, figsize=(10, 2))

for ax, p, t in zip(
    axs,
    [1 / 12 * R.expand_dims("j"), thf, ekf, zaf + dd, -alpha.expand_dims("j")],
    [r"$R$", "THF", "EKM", "ZAF+DD", r"$-\alpha$"],
):

    ax.plot(p.month, 12 * (p).isel(j=ji, year=0))
    ax.plot(p.month, 12 * (p).isel(j=ji, year=10))
    ax.plot(p.month, 12 * (p).isel(j=ji, year=-1))
    ax.axhline(0, ls="--", c="k", lw=0.8)
    ax.set_title(t)

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])
plt.show()

In [ ]:
delta = lambda x: x.isel(year=-1) - x.isel(year=0)
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(R.month, delta(R), label=r"$R$", c="k", lw=2)
ax.plot(R.month, 12 * delta(thf + ekf).isel(j=0), label=r"THF+EKF")
ax.plot(R.month, 12 * delta(thf + ekf + alpha + dd + zaf).isel(j=0), label=r"all")
ax.legend(prop=dict(size=10), loc=(1, 0.3))
ax.axhline(0, c="k", ls="--", lw=0.8)
plt.show()

## Scratch

#### estimate with Greens' fn

In [ ]:
## get data and lagged data
X0 = Th_normed.isel(time=slice(None, -1))
X1 = Th_normed.isel(time=slice(1, None))
X1 = X1.assign_coords({"time": X0.time})

## rename vars in X1
names = {v: f"{v}_plus" for v in list(X1)}
X1 = X1.rename({v: f"{v}_plus" for v in list(X1)})

## merge
X = xr.merge([X0, X1])

In [ ]:
## compute Green's fn
G = (
    regress_harm(X, x_vars=list(X0), y_vars=list(X1))
    .squeeze(
        drop=True,
    )
    .rename({"ell": "j"})
)


def get_L(G):
    """compute linear operator from Green's fn"""

    ## test: convert to array
    G_array = G.to_dataarray(dim="y").rename({"j": "x"})
    G_array = G_array.transpose("y", "x")

    ## eig decomp
    s, v = np.linalg.eig(G_array.values)

    ## check recon works
    assert np.allclose(G_array.values, v @ np.diag(s) @ np.linalg.inv(v))

    ## get operator
    L = xr.zeros_like(G_array)
    L_vals = v @ np.diag(12 * np.log(s)) @ np.linalg.inv(v)
    L.values = L_vals.real

    return L


L_v2 = []
for year in tqdm.tqdm(G.year):

    ## empty list to hold operator for year
    L_year = []

    ## loope thru months
    for month in G.month:
        L_year.append(get_L(G.sel(year=year, month=month)))

    ## concatenate for year
    L_year = xr.concat(L_year, dim=G.month)
    L_v2.append(L_year)

L_v2 = xr.concat(L_v2, dim=G.year)

R_v2 = L_v2.isel(y=0, x=0)
eps_v2 = -L_v2.isel(y=1, x=1)
F1_v2 = L_v2.isel(y=0, x=1)
F2_v2 = -L_v2.isel(y=1, x=0)

In [ ]:
# p0 = R-eps
# p1 = R_v2-eps_v2
p0 = R
p1 = R_v2

fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))
# plt.plot(R.mean("month"))
axs[0].plot(R.year, p0.mean("month"))
axs[0].plot(R.year, p1.mean("month"))

for ax, yi in zip(axs[1:], [0, -1]):
    ax.plot(R.month, p0.isel(year=yi))
    ax.plot(R.month, p1.isel(year=yi))
    ax.axvspan(xmin=2.8, xmax=4.2, alpha=0.2)
    ax.axvspan(xmin=5.8, xmax=7.2, alpha=0.2)